# Freelancer Job Recommender — Full Walkthrough

This notebook demonstrates every enhancement made to the original system:
1. Data cleaning & enrichment
2. Incremental embedding with cache
3. Qdrant ANN indexing
4. Hybrid scoring (semantic + structured + geo)
5. Proper IR evaluation (NDCG, Precision@K)
6. Reverse matching (job → freelancers)
7. Model benchmarking

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from utils.logging_setup import setup_logging
setup_logging()

## 1 · Data loading & cleaning

In [ ]:
from data.preprocessor import DataPreprocessor
from config.settings import FREELANCERS_CSV, JOBS_CSV

prep = DataPreprocessor(FREELANCERS_CSV, JOBS_CSV)
df_f, df_j = prep.load_and_clean()

print(f'Freelancers: {len(df_f):,}  |  Jobs: {len(df_j):,}')
print('\nFreelancer columns:', df_f.columns.tolist())
print('\nNew parsed fields:')
df_f[['name','rate_usd','earnings_usd','feedback_score','jobs_done']].head(5)

In [ ]:
# Verify skills_cleaned no longer has empty tokens
print('Sample cleaned skills:')
for s in df_f['skills_cleaned'].head(3):
    print(' ', s)

In [ ]:
# Verify enriched_text is clean (no duplication)
print('Sample enriched_text (first 200 chars):')
print(df_f['enriched_text'].iloc[0][:200])
print('---')
print('Job enriched_text with reputation signal:')
print(df_j['enriched_text'].iloc[0][:200])

## 2 · Embedding with incremental cache

In [ ]:
from models.embedding_engine import EmbeddingEngine

embedder = EmbeddingEngine()
cache = embedder.load_or_compute(df_f, df_j)

f_emb = cache['freelancer_embeddings']
j_emb = cache['job_embeddings']

print(f'Freelancer embeddings: {f_emb.shape}')
print(f'Job embeddings:        {j_emb.shape}')

## 3 · Qdrant indexing (ANN)

In [ ]:
from models.vector_store import VectorStore

store = VectorStore()   # ':memory:' by default
store.index_jobs(j_emb, df_j)
store.index_freelancers(f_emb, df_f)
print('Qdrant index ready')

## 4 · Hybrid scoring demo

In [ ]:
from models.scoring_engine import ScoringEngine

scorer = ScoringEngine()
freelancer = df_f.iloc[0]

raw = store.search_jobs(f_emb[0], top_n=20)
matches = scorer.rank(raw, freelancer)

print(f'Freelancer: {freelancer["name"]} | {freelancer["job_title"]}')
print(f'Rate: ${freelancer["rate_usd"]}  Feedback: {freelancer["feedback_score"]:.0%}\n')
for m in matches[:10]:
    print(m.summary())

## 5 · Full engine (one-liner)

In [ ]:
from models.recommendation_engine import RecommendationEngine

engine = RecommendationEngine()
engine.build()

In [ ]:
# Freelancer → Jobs
matches = engine.recommend_jobs(freelancer_index=1257, top_n=10)
engine.print_recommendations(matches, title='Jobs for freelancer #1257')

In [ ]:
# Reverse: Job → Freelancers  (new capability)
matches = engine.recommend_freelancers(job_index=0, top_n=10)
engine.print_recommendations(matches, title='Best freelancers for job #0')

## 6 · Proper evaluation (NDCG, Precision@K)

In [ ]:
from evaluation.evaluation_engine import EvaluationEngine

evaluator = EvaluationEngine(engine.df_freelancers, engine.df_jobs,
                              engine._f_emb, engine._j_emb)
summary = evaluator.evaluate(top_n=10, sample_freelancers=200, sample_jobs=500)
summary

In [ ]:
evaluator.plot_distribution(top_n=10)

## 7 · Model benchmarking (optional — downloads models)

In [ ]:
# Uncomment to run (downloads several models — takes a few minutes)
# models_to_compare = [
#     'BAAI/bge-base-en-v1.5',   # current
#     'BAAI/bge-large-en-v1.5',  # larger / slower
#     'intfloat/e5-base-v2',
#     'all-MiniLM-L6-v2',        # original
# ]
# bench = embedder.benchmark_models(df_f, df_j, models_to_compare)
# evaluator.compare_models(bench)
# bench